# EpiSentinel — Weekly Cases Pipeline
**Purpose:** Build an enhanced weekly_cases.csv optimized for LSTM outbreak detection.

**Input:** master_dataset.csv  
**Output:** weekly_cases_enhanced.csv

---

## Step 1: Load Master Dataset

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

MASTER_PATH = r'C:\Users\ahdsy\Downloads\synthea-master\synthea-master\output\master_dataset.csv'
OUTPUT_PATH = r'C:\Users\ahdsy\Downloads\synthea-master\synthea-master\output\weekly_cases_enhanced.csv'

print('Loading master dataset...')
df = pd.read_csv(MASTER_PATH, parse_dates=['VISIT_DATE'])
print(f'Loaded: {len(df):,} records')
print(f'Date range: {df["VISIT_DATE"].min()} to {df["VISIT_DATE"].max()}')

Loading master dataset...
Loaded: 6,175,277 records
Date range: 2000-01-01 00:16:23 to 2026-07-10 10:00:13


## Step 2: Add ISO Week Start Date

In [2]:
# Convert VISIT_DATE to start of ISO week (Monday)
# This ensures consistent weekly grouping
df['VISIT_WEEK_START'] = df['VISIT_DATE'].dt.to_period('W').apply(lambda r: r.start_time)

print('Sample week starts:')
print(df[['VISIT_DATE', 'VISIT_WEEK_START']].head(5).to_string())

Sample week starts:
           VISIT_DATE VISIT_WEEK_START
0 2000-01-01 00:16:23       1999-12-27
1 2000-01-01 00:27:01       1999-12-27
2 2000-01-01 00:37:48       1999-12-27
3 2000-01-01 01:07:39       1999-12-27
4 2000-01-01 01:10:04       1999-12-27


## Step 3: Aggregate Weekly Cases per City + Disease

In [3]:
print('Aggregating weekly cases...')

weekly = df.groupby(['COUNTRY', 'CITY', 'DISEASE', 'VISIT_WEEK_START']).agg(
    case_count        = ('PATIENT', 'count'),
    outbreak_cases    = ('IS_OUTBREAK', 'sum'),
    avg_age           = ('AGE_AT_VISIT', 'mean'),
    avg_comorbidity   = ('COMORBIDITY_COUNT', 'mean'),
    VISIT_YEAR        = ('VISIT_YEAR', 'first'),
    VISIT_MONTH       = ('VISIT_MONTH', 'first'),
    SEASON            = ('SEASON', 'first'),
    LAT               = ('LAT', 'first'),
    LON               = ('LON', 'first'),
).reset_index()

weekly = weekly.sort_values(['COUNTRY', 'CITY', 'DISEASE', 'VISIT_WEEK_START']).reset_index(drop=True)

print(f'Weekly records before gap fill: {len(weekly):,}')
print(f'Unique City+Disease combinations: {weekly.groupby(["CITY","DISEASE"]).ngroups:,}')

Aggregating weekly cases...
Weekly records before gap fill: 709,129
Unique City+Disease combinations: 751


## Step 4: Fill Missing Weeks (No Gaps in Time Series)
Critical: LSTM breaks if there are gaps in the sequence.
Every City+Disease combination must have a row for every week in its date range.

In [4]:
print('Filling missing weeks...')

filled_groups = []

for (country, city, disease), group in weekly.groupby(['COUNTRY', 'CITY', 'DISEASE']):
    group = group.set_index('VISIT_WEEK_START').sort_index()

    # Create full weekly date range from first to last week
    full_range = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq='W-MON'
    )

    # Reindex to fill gaps with 0 cases
    group = group.reindex(full_range, fill_value=0)
    group.index.name = 'VISIT_WEEK_START'
    group = group.reset_index()

    # Fill static columns
    group['COUNTRY'] = country
    group['CITY']    = city
    group['DISEASE'] = disease

    # Fill VISIT_YEAR and VISIT_MONTH from the week start date
    group['VISIT_YEAR']  = group['VISIT_WEEK_START'].dt.year
    group['VISIT_MONTH'] = group['VISIT_WEEK_START'].dt.month

    # Fill SEASON from month
    def get_season(month):
        if   month in [3, 4, 5]:   return 'Long_Rains'
        elif month in [6, 7, 8]:   return 'Dry_Season'
        elif month in [9, 10, 11]: return 'Short_Rains'
        else:                      return 'Cool_Dry'

    group['SEASON'] = group['VISIT_MONTH'].apply(get_season)

    # Fill LAT/LON (forward fill from original rows)
    for col in ['LAT', 'LON']:
        if group[col].eq(0).any():
            group[col] = group[col].replace(0, np.nan).ffill().bfill()

    filled_groups.append(group)

weekly = pd.concat(filled_groups, ignore_index=True)
weekly = weekly.sort_values(['COUNTRY', 'CITY', 'DISEASE', 'VISIT_WEEK_START']).reset_index(drop=True)

print(f'Weekly records after gap fill: {len(weekly):,}')
print(f'Zero-case weeks added: {(weekly["case_count"] == 0).sum():,}')

Filling missing weeks...
Weekly records after gap fill: 983,425
Zero-case weeks added: 274,296


## Step 5: Add Lag Features
Gives the LSTM memory of what happened in previous weeks.

In [5]:
print('Adding lag features...')

# Group by city+disease for correct lag calculation
weekly = weekly.sort_values(['CITY', 'DISEASE', 'VISIT_WEEK_START'])

for lag in [1, 2, 3, 4]:
    weekly[f'lag_{lag}_cases'] = (
        weekly.groupby(['CITY', 'DISEASE'])['case_count']
        .shift(lag)
        .fillna(0)
    )

print('Lag features added: lag_1_cases, lag_2_cases, lag_3_cases, lag_4_cases')
print(weekly[['CITY', 'DISEASE', 'VISIT_WEEK_START', 'case_count',
              'lag_1_cases', 'lag_2_cases', 'lag_3_cases', 'lag_4_cases']].head(8).to_string())

Adding lag features...
Lag features added: lag_1_cases, lag_2_cases, lag_3_cases, lag_4_cases
         CITY   DISEASE VISIT_WEEK_START  case_count  lag_1_cases  lag_2_cases  lag_3_cases  lag_4_cases
697237  Abuja  COVID-19       2018-12-31           6          0.0          0.0          0.0          0.0
697238  Abuja  COVID-19       2019-01-07           4          6.0          0.0          0.0          0.0
697239  Abuja  COVID-19       2019-01-14           4          4.0          6.0          0.0          0.0
697240  Abuja  COVID-19       2019-01-21           4          4.0          4.0          6.0          0.0
697241  Abuja  COVID-19       2019-01-28           0          4.0          4.0          4.0          6.0
697242  Abuja  COVID-19       2019-02-04           9          0.0          4.0          4.0          4.0
697243  Abuja  COVID-19       2019-02-11           2          9.0          0.0          4.0          4.0
697244  Abuja  COVID-19       2019-02-18           5          2.0 

## Step 6: Add Rolling Statistics
Different window sizes capture different disease spread speeds.

In [6]:
print('Adding rolling statistics...')

grouped = weekly.groupby(['CITY', 'DISEASE'])['case_count']

# Rolling averages — 2, 3, 4 week windows
weekly['cases_2wk_avg'] = grouped.transform(lambda x: x.rolling(2, min_periods=1).mean()).round(2)
weekly['cases_3wk_avg'] = grouped.transform(lambda x: x.rolling(3, min_periods=1).mean()).round(2)
weekly['cases_4wk_avg'] = grouped.transform(lambda x: x.rolling(4, min_periods=1).mean()).round(2)

# Rolling std — spike detector
weekly['cases_4wk_std'] = grouped.transform(lambda x: x.rolling(4, min_periods=1).std().fillna(0)).round(2)

print('Rolling features added: cases_2wk_avg, cases_3wk_avg, cases_4wk_avg, cases_4wk_std')

Adding rolling statistics...
Rolling features added: cases_2wk_avg, cases_3wk_avg, cases_4wk_avg, cases_4wk_std


## Step 7: Add Trend Features
Helps the model detect acceleration in case growth — key early warning signal.

In [7]:
print('Adding trend features...')

# Week-over-week difference
weekly['cases_diff'] = (
    weekly.groupby(['CITY', 'DISEASE'])['case_count']
    .diff()
    .fillna(0)
    .round(2)
)

# Growth rate (week-over-week % change)
# Avoid division by zero with clip
weekly['pct_change_wow'] = (
    weekly.groupby(['CITY', 'DISEASE'])['case_count']
    .pct_change()
    .fillna(0)
    .replace([np.inf, -np.inf], 0)
    .round(4)
)

# Growth rate relative to 4-week average (detects acceleration vs baseline)
weekly['growth_rate'] = (
    (weekly['case_count'] - weekly['cases_4wk_avg'])
    / weekly['cases_4wk_avg'].clip(lower=1)
).round(4)

print('Trend features added: cases_diff, pct_change_wow, growth_rate')

Adding trend features...
Trend features added: cases_diff, pct_change_wow, growth_rate


## Step 8: Add Outbreak Label
label_outbreak_week = 1 if any outbreak case occurred that week.

In [8]:
# Binary label: 1 if any outbreak case that week
weekly['label_outbreak_week'] = (weekly['outbreak_cases'] > 0).astype(int)

outbreak_weeks = weekly['label_outbreak_week'].sum()
total_weeks = len(weekly)
print(f'Outbreak weeks: {outbreak_weeks:,} ({outbreak_weeks/total_weeks*100:.2f}%)')
print(f'Normal weeks  : {total_weeks - outbreak_weeks:,} ({(total_weeks-outbreak_weeks)/total_weeks*100:.2f}%)')

Outbreak weeks: 3,560 (0.36%)
Normal weeks  : 979,865 (99.64%)


## Step 9: Final Column Order & Save

In [9]:
# Final column order — logical grouping
final_cols = [
    # Identifiers
    'COUNTRY', 'CITY', 'DISEASE', 'VISIT_WEEK_START',
    # Temporal
    'VISIT_YEAR', 'VISIT_MONTH', 'SEASON',
    # Geography
    'LAT', 'LON',
    # Core case counts
    'case_count', 'outbreak_cases',
    # Patient features
    'avg_age', 'avg_comorbidity',
    # Lag features
    'lag_1_cases', 'lag_2_cases', 'lag_3_cases', 'lag_4_cases',
    # Rolling averages
    'cases_2wk_avg', 'cases_3wk_avg', 'cases_4wk_avg', 'cases_4wk_std',
    # Trend features
    'cases_diff', 'pct_change_wow', 'growth_rate',
    # Target
    'label_outbreak_week',
]

weekly = weekly[final_cols]


## Save Final weekly_cases_enhanced.csv

In [10]:
# ===========================================================
# SAVE — Must run after all transformations are complete
# ===========================================================
print(f'Saving to {OUTPUT_PATH}...')
weekly.to_csv(OUTPUT_PATH, index=False)

print(f'\n✅ weekly_cases_enhanced.csv saved!')
print(f'Shape  : {weekly.shape}')
print(f'Columns: {len(weekly.columns)}')
print(f'\nColumn list:')
for col in weekly.columns:
    print(f'  {col}')


Saving to C:\Users\ahdsy\Downloads\synthea-master\synthea-master\output\weekly_cases_enhanced.csv...

✅ weekly_cases_enhanced.csv saved!
Shape  : (983425, 25)
Columns: 25

Column list:
  COUNTRY
  CITY
  DISEASE
  VISIT_WEEK_START
  VISIT_YEAR
  VISIT_MONTH
  SEASON
  LAT
  LON
  case_count
  outbreak_cases
  avg_age
  avg_comorbidity
  lag_1_cases
  lag_2_cases
  lag_3_cases
  lag_4_cases
  cases_2wk_avg
  cases_3wk_avg
  cases_4wk_avg
  cases_4wk_std
  cases_diff
  pct_change_wow
  growth_rate
  label_outbreak_week


## Step 10: Validation

In [11]:
print('='*55)
print('VALIDATION REPORT')
print('='*55)

passed = 0
failed = 0

def check(name, condition, detail=''):
    global passed, failed
    status = '✅ PASS' if condition else '❌ FAIL'
    if condition: passed += 1
    else: failed += 1
    print(f'{status} | {name}')
    if detail: print(f'       {detail}')

# Load master for cross-checks
master = pd.read_csv(MASTER_PATH)

print('\n--- 1. Case Count Consistency ---')
weekly_total = weekly['case_count'].sum()
master_total = len(master)
check('Sum of case_count == total master records',
      weekly_total == master_total,
      f'Weekly sum: {weekly_total:,} | Master rows: {master_total:,}')

print('\n--- 2. Outbreak Cases Consistency ---')
weekly_outbreaks = weekly['outbreak_cases'].sum()
master_outbreaks = master['IS_OUTBREAK'].sum()
check('Sum of outbreak_cases == IS_OUTBREAK sum in master',
      weekly_outbreaks == master_outbreaks,
      f'Weekly: {weekly_outbreaks:,} | Master: {master_outbreaks:,}')

print('\n--- 3. No Gaps in Time Series ---')
gaps_found = 0
for (city, disease), group in weekly.groupby(['CITY', 'DISEASE']):
    dates = pd.to_datetime(group['VISIT_WEEK_START']).sort_values()
    diffs = dates.diff().dropna()
    non_weekly = (diffs != pd.Timedelta('7 days')).sum()
    gaps_found += non_weekly

check('No gaps in weekly time series',
      gaps_found == 0,
      f'Gaps found: {gaps_found}')

print('\n--- 4. No Missing Values in Key Columns ---')
key_cols = ['case_count', 'lag_1_cases', 'lag_2_cases',
            'cases_3wk_avg', 'pct_change_wow', 'label_outbreak_week']
for col in key_cols:
    nulls = weekly[col].isna().sum()
    check(f'No nulls in {col}',
          nulls == 0,
          f'Nulls: {nulls}')

print('\n--- 5. Label Balance ---')
outbreak_pct = weekly['label_outbreak_week'].mean() * 100
check('Outbreak weeks between 0.1% and 5%',
      0.1 <= outbreak_pct <= 5,
      f'Outbreak weeks: {outbreak_pct:.2f}%')

print('\n--- 6. Lag Features Correct ---')
sample = weekly[weekly['CITY']=='Cairo'][weekly['DISEASE']=='Malaria'].head(5)
lag_correct = all(
    sample['lag_1_cases'].iloc[i] == sample['case_count'].iloc[i-1]
    for i in range(1, len(sample))
)
check('lag_1_cases == previous week case_count',
      lag_correct)

print('\n--- 7. Rolling Averages Correct ---')
check('cases_3wk_avg is always <= max(case_count) * 1.1',
      (weekly['cases_3wk_avg'] <= weekly['case_count'].max() * 1.1).all())

print('\n--- 8. No Infinite Values ---')
numeric_cols = weekly.select_dtypes(include=[np.number]).columns
inf_count = np.isinf(weekly[numeric_cols]).sum().sum()
check('No infinite values in numeric columns',
      inf_count == 0,
      f'Infinite values: {inf_count}')

print('\n--- 9. SEASON Values Valid ---')
valid_seasons = {'Long_Rains', 'Short_Rains', 'Dry_Season', 'Cool_Dry'}
invalid_seasons = set(weekly['SEASON'].unique()) - valid_seasons
check('All SEASON values are valid',
      len(invalid_seasons) == 0,
      f'Invalid seasons: {invalid_seasons}')

print('\n--- 10. Date Range ---')
min_week = pd.to_datetime(weekly['VISIT_WEEK_START']).min()
max_year = pd.to_datetime(weekly['VISIT_WEEK_START']).dt.year.max()
check('Earliest week start is within 7 days of 2000-01-01 (ISO week boundary)',
      min_week >= pd.Timestamp('1999-12-25') and max_year <= 2026,
      f'Earliest week start: {min_week.date()} | Max year: {max_year}')

print('\n' + '='*55)
print('SUMMARY')
print('='*55)
print(f'✅ Passed  : {passed}')
print(f'❌ Failed  : {failed}')
print(f'Total     : {passed + failed}')

print('\nFinal dataset stats:')
print(f'  Rows    : {len(weekly):,}')
print(f'  Columns : {len(weekly.columns)}')
print(f'  Cities  : {weekly["CITY"].nunique()}')
print(f'  Diseases: {weekly["DISEASE"].nunique()}')
print(f'  Weeks   : {weekly["VISIT_WEEK_START"].nunique():,}')

if failed == 0:
    print('\n🎉 All checks passed — weekly_cases_enhanced.csv is ready for LSTM training!')
else:
    print(f'\n⚠️ {failed} check(s) failed — review before training.')

VALIDATION REPORT

--- 1. Case Count Consistency ---
✅ PASS | Sum of case_count == total master records
       Weekly sum: 6,175,277 | Master rows: 6,175,277

--- 2. Outbreak Cases Consistency ---
✅ PASS | Sum of outbreak_cases == IS_OUTBREAK sum in master
       Weekly: 286,584 | Master: 286,584

--- 3. No Gaps in Time Series ---
✅ PASS | No gaps in weekly time series
       Gaps found: 0

--- 4. No Missing Values in Key Columns ---
✅ PASS | No nulls in case_count
       Nulls: 0
✅ PASS | No nulls in lag_1_cases
       Nulls: 0
✅ PASS | No nulls in lag_2_cases
       Nulls: 0
✅ PASS | No nulls in cases_3wk_avg
       Nulls: 0
✅ PASS | No nulls in pct_change_wow
       Nulls: 0
✅ PASS | No nulls in label_outbreak_week
       Nulls: 0

--- 5. Label Balance ---
✅ PASS | Outbreak weeks between 0.1% and 5%
       Outbreak weeks: 0.36%

--- 6. Lag Features Correct ---
✅ PASS | lag_1_cases == previous week case_count

--- 7. Rolling Averages Correct ---
✅ PASS | cases_3wk_avg is always <= ma